# Step 05 — Inspect and filter the contracted barcode graph

Loads the graph from step 04, extracts its largest connected component, shows diagnostics, and saves a filtered graph (minimum edge weight, maximum degree, leaf stripping, LCC).

## Imports

In [ ]:
import time
import math
import random
import os
import heapq
import numpy as np
from graph_tool import Graph, GraphView
from graph_tool.draw import sfdp_layout, random_layout
from graph_tool.topology import label_components, label_largest_component
import graph_tool as gt
import matplotlib.pyplot as plt

print("graph_tool Version: ", print(gt.__version__))
print("OpenMP Enabled: ", gt.openmp_enabled())

if gt.openmp_enabled():
    print("Number of threads before: ", gt.openmp_get_num_threads())
    gt.openmp_set_num_threads(1)
    print("Number of threads after: ", gt.openmp_get_num_threads())

## Parameters
Set the project, sample and the suffix of the graph produced by step 04.

In [ ]:
# Parameters
project_name = "260921Jac"
scratch_prefix = os.path.join(os.environ.get("SPAN_DATA_ROOT", "./orcd"), "scratch", project_name) + "/"

SAMPLE_ID = "M1"

# Sample radius; used only for the "nm per pixel" printout
total_radius = 3.75

###########
print("Project Name: ", project_name)
print("Sample ID: ", SAMPLE_ID)

## Load graph and basic statistics

In [ ]:
# Output of step 04 (04_neighbor_contraction.py)
suffix = "all_Hamming1_Hamming2_Lev3_contractedK2"

filename = scratch_prefix + SAMPLE_ID + "_barcodesGraph_" + suffix + ".gt.gz"

print("Loading: ", filename)

t0 = time.perf_counter()
g = gt.load_graph(filename)
elapsed_min = (time.perf_counter() - t0) / 60.0

print(f"Loaded in {elapsed_min:.2f} minutes.")

In [ ]:
num_nodes = g.num_vertices()
num_edges = g.num_edges()

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of edges: {num_edges:,}")

avg_degree = (2 * num_edges / num_nodes) if num_nodes > 0 else 0.0   # for an undirected graph

print(f"Average degree: {avg_degree:.2f}")


print(f"Directed: {g.is_directed()}")


print("nm per pixel: ", round(2 * np.sqrt(total_radius**2 * np.pi * 10**6 / num_nodes / np.pi), 2), "nm")

ep_w = g.edge_properties.get("weight", None)

total_w = int(np.sum(ep_w.a)) if ep_w is not None else g.num_edges()
print(f"Total edge weight: {total_w:,}")

In [ ]:
# Connected components + top 5 largest connected components ---
comp, hist = label_components(g, directed=False)  # respects view
num_cc = int(len(hist))

print(f"Connected components (filtered): {num_cc:,}")

if num_cc:
    top_n = min(5, num_cc)
    top_ids = np.argsort(hist.a if hasattr(hist, "a") else np.array(hist))[::-1][:top_n]

    print(f"Top {top_n} connected components:")
    for rank, cc_id in enumerate(top_ids, start=1):
        cc_size = int(hist[cc_id])
        print(f"  {rank}. id={int(cc_id)}  size={cc_size:,} vertices")
else:
    print("No connected components found.")

## Restrict to the largest connected component (LCC)

In [ ]:
# Largest connected component (treat as undirected)
lcc_vfilt = label_largest_component(g, directed=False)

# View + prune into a new standalone graph, then replace g
g = Graph(GraphView(g, vfilt=lcc_vfilt), prune=True)

In [ ]:
num_nodes = g.num_vertices()
num_edges = g.num_edges()

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of edges: {num_edges:,}")


print("nm per pixel: ", round(2 * np.sqrt(total_radius**2 * np.pi * 10**6 / num_nodes / np.pi), 2), "nm")

ep_w = g.edge_properties.get("weight", None)

total_w = int(np.sum(ep_w.a)) if ep_w is not None else g.num_edges()
print(f"Total edge weight: {total_w:,}")

## Diagnostics (optional)
Degree distributions, GC content vs. degree, and how the LCC size changes with degree and edge-weight thresholds. These cells only inspect the graph; they help choose `min_w` and `max_deg` below.

In [ ]:
# ---- compute (unweighted) degree for every vertex ----
# total_degree() works for directed/undirected; for undirected it's just the usual degree.
deg = g.get_total_degrees(g.get_vertices()).astype(np.int64)
N = int(deg.size)

def _count(mask):
    return int(np.count_nonzero(mask))

def _pct(count):
    return 100.0 * count / N if N else 0.0

# ---- bins you asked for ----
bins = [
    ("deg = 0",            (deg == 0)),
    ("deg = 1",            (deg == 1)),
    ("deg = 2",            (deg == 2)),
    ("deg = 3",            (deg == 3)),
    ("deg = 4",            (deg == 4)),
    ("deg = 5",            (deg == 5)),
    ("deg = 6",            (deg == 6)),
    ("deg = 7",            (deg == 7)),
    ("deg = 8",            (deg == 8)),
    ("deg = 9",            (deg == 9)),
    ("deg = 10",        (deg == 10)),
    ("deg 11–20",          ((deg >= 11) &(deg <= 20))),
    ("deg 21–30",          ((deg >= 21) & (deg <= 30))),
    ("deg 31–40",          ((deg >= 31) & (deg <= 40))),
    ("deg 41–50",          ((deg >= 41) & (deg <= 50))),
    ("deg 51–100",         ((deg >= 51) & (deg <= 100))),
    ("deg 101–200",        ((deg >= 101) & (deg <= 200))),
    ("deg 201–300",        ((deg >= 201) & (deg <= 300))),
]

print(f"Total vertices: {N:,}")
print()

for label, mask in bins:
    c = _count(mask)
    print(f"{label:12s}: {c:>10,}  ({_pct(c):6.2f}%)")
    
print()
mask_2_30 = (deg >= 2) & (deg <= 30)
c_2_30 = int(np.count_nonzero(mask_2_30))
print(f"deg 2–30     : {c_2_30:>10,}  ({100.0*c_2_30/N:6.2f}%)")
print()

# ---- >300 degrees + list them ----
mask_hi = deg > 300
c_hi = _count(mask_hi)
print()
print(f"deg > 300   : {c_hi:>10,}  ({_pct(c_hi):6.2f}%)")

if c_hi > 0:
    # show degrees (and vertex ids) for high-degree vertices
    vids = g.get_vertices()[mask_hi]
    degrees = deg[mask_hi]

    # sort by degree descending
    order = np.argsort(degrees)[::-1]
    vids = vids[order]
    degrees = degrees[order]

    print("\nVertices with deg > 300 (sorted):")
    # If you have a barcode property, show it too
    has_barcode = "barcode" in g.vp

    for vid, d in zip(vids, degrees):
        if has_barcode:
            print(f"  v={int(vid):>10d}  deg={int(d):>6d}  barcode={g.vp.barcode[g.vertex(int(vid))]}")
        else:
            print(f"  v={int(vid):>10d}  deg={int(d):>6d}")

In [ ]:
# degrees for all vertices
deg = g.get_total_degrees(g.get_vertices()).astype(np.int64)

# keep only degrees 1..30
deg_clip = deg[(deg >= 2) & (deg <= 40)]

# counts for each degree 1..30 (include zeros)
bins = np.arange(1, 42)  # edges: 1..31
counts, edges = np.histogram(deg_clip, bins=bins)
xs = np.arange(1, 41)

plt.figure(figsize=(12, 7))
plt.bar(xs, counts)
plt.xlabel("Degree")
plt.ylabel("Number of vertices")
plt.title("Vertex degree histogram (degrees 2–30; vertices with degree > 30 ignored)")
plt.xticks(xs)
plt.grid(True, axis="y")
plt.show()

# # Optional: print counts table
# for d, c in zip(xs, counts):
#     print(f"deg={d:2d}: {int(c):,}")


In [ ]:
# ------------------------------------------------------------------
# Degree distribution AFTER the edge-weight filter (w >= min_w)
# ------------------------------------------------------------------
min_w      = 2          # edge weight threshold
weight_ep  = "weight"
do_lcc     = True      # True -> also restrict to the LCC, as the pipeline does

w = g.ep[weight_ep]

# Edge filter: weight >= min_w  (vertices are all still present, just possibly isolated)
ef = g.new_ep("bool")
ef.a = (w.a >= min_w)
gv = gt.GraphView(g, efilt=ef, directed=False)

if do_lcc:
    lcc_vfilt = label_largest_component(gv, directed=False)
    gv = gt.GraphView(gv, vfilt=lcc_vfilt)

print(f"After w >= {min_w}"
      f"{' + LCC' if do_lcc else ''}: "
      f"V={gv.num_vertices():,}  E={gv.num_edges():,}")

# Degrees in the FILTERED view
deg = gv.get_total_degrees(gv.get_vertices()).astype(np.int64)
N = int(deg.size)

# ---------------- Histogram, degrees 2..40 ----------------
deg_clip = deg[(deg >= 2) & (deg <= 40)]

bins = np.arange(1, 42)      # edges 1..41
counts, edges = np.histogram(deg_clip, bins=bins)
xs = np.arange(1, 41)

plt.figure(figsize=(12, 7))
plt.bar(xs, counts)
plt.xlabel("Degree")
plt.ylabel("Number of vertices")
plt.title(f"Vertex degree histogram after w >= {min_w} "
          f"(degrees 2–40; degree > 40 shown in table below)")
plt.xticks(xs)
plt.grid(True, axis="y")
plt.show()

# ---------------- Tail counts ----------------
tail_bins = [
    ("deg = 0",         (deg == 0)),
    ("deg = 1",         (deg == 1)),
    ("2 <= deg <= 40",  (deg >= 2)   & (deg <= 40)),
    ("40 < deg <= 100", (deg > 40)   & (deg <= 100)),
    ("100 < deg <= 200",(deg > 100)  & (deg <= 200)),
    ("deg > 200",       (deg > 200)),
]

print(f"\nTotal vertices in view: {N:,}")
print(f"{'bin':<18}{'count':>12}{'% of V':>10}")
print("-" * 40)
for label, mask in tail_bins:
    c = int(np.count_nonzero(mask))
    print(f"{label:<18}{c:>12,}{(100.0 * c / N if N else 0):>9.3f}%")
print("-" * 40)
print(f"max degree: {int(deg.max()) if N else 0:,}   "
      f"mean degree: {deg.mean() if N else 0:.2f}")

# How many vertices the max_deg=30 cap would remove at this threshold
for cap in (30, 40):
    n_cut = int(np.count_nonzero(deg > cap))
    print(f"vertices with deg > {cap}: {n_cut:,} "
          f"({100.0 * n_cut / N if N else 0:.3f}% of V)")

# Optional: per-degree counts table for 2..40
# for d, c in zip(xs, counts):
#     if d >= 2:
#         print(f"deg={d:2d}: {int(c):,}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_gc_vs_log_degree(g, subset="L", barcode_vp="barcode", point_size=4, alpha=0.15):
    if subset not in {"L", "R"}:
        raise ValueError("subset must be 'L' or 'R'")

    barcode = g.vp[barcode_vp]
    verts = g.get_vertices()
    deg = g.get_total_degrees(verts).astype(np.int64)

    x_vals = []
    y_vals = []

    gc_deg_3_30 = []
    gc_deg_gt_30 = []
    gc_deg_gt_100 = []

    for v, d in zip(verts, deg):
        if d < 2:
            continue

        bc = barcode[v]
        if not bc.startswith(subset):
            continue

        seq = bc[1:]
        if not seq:
            continue

        gc_pct = 100.0 * sum(b in {"G", "C"} for b in seq) / len(seq)

        x_vals.append(np.log10(d))
        y_vals.append(gc_pct)

        if 3 <= d <= 30:
            gc_deg_3_30.append(gc_pct)
        if d > 30:
            gc_deg_gt_30.append(gc_pct)
        if d > 100:
            gc_deg_gt_100.append(gc_pct)

    x_vals = np.array(x_vals)
    y_vals = np.array(y_vals)

    plt.figure(figsize=(10, 7))
    plt.scatter(x_vals, y_vals, s=point_size, alpha=alpha)
    plt.xlabel("log10(Degree)")
    plt.ylabel("GC percentage")
    plt.title(f"{subset} barcodes: GC% vs log10(degree)")
    plt.ylim(0, 100)
    plt.grid(True, alpha=0.3)
    plt.show()

    avg_3_30 = np.mean(gc_deg_3_30) if gc_deg_3_30 else np.nan
    avg_gt_30 = np.mean(gc_deg_gt_30) if gc_deg_gt_30 else np.nan
    avg_gt_100 = np.mean(gc_deg_gt_100) if gc_deg_gt_100 else np.nan

    print(f"{subset} barcodes, degrees 3-30: average GC% = {avg_3_30:.2f}")
    print(f"{subset} barcodes, degrees >30: average GC% = {avg_gt_30:.2f}")
    print(f"{subset} barcodes, degrees >100: average GC% = {avg_gt_100:.2f}")

In [ ]:
plot_gc_vs_log_degree(g, subset="L", point_size=6, alpha=0.1)

In [ ]:
plot_gc_vs_log_degree(g, subset="R", point_size=6, alpha=0.1)

In [ ]:
K = 30

# --- 1) Build a zero-copy GraphView that excludes high-degree vertices ---
# (degree computed on the current graph g; assumes g is undirected or you want total degree)
deg = g.get_total_degrees(g.get_vertices()).astype(np.int64)   # numpy array, len = g.num_vertices()
vfilt = g.new_vertex_property("bool")
vfilt.a = (deg <= K)  # keep only vertices with degree <= K

gv = gt.GraphView(g, vfilt=vfilt)  # NO COPY

print(f"Original graph: V={g.num_vertices():,}  E={g.num_edges():,}")
print(f"Filtered view (deg <= {K}): V={gv.num_vertices():,}  E={gv.num_edges():,}")

In [ ]:

# --- 2) Total edge weight in the filtered view ---
if "weight" in gv.ep:
    w = gv.ep.weight

    # Fast method (uses an edges array; can be memory-heavy on very large graphs)
    edges_w = gv.get_edges([w])  # columns: src, dst, weight  (respects the view)
    total_w = float(edges_w[:, 2].sum()) if edges_w.size else 0.0
    print(f"Total edge weight (filtered): {total_w:,.0f}")

    # If you ever hit memory issues with get_edges, use this slower streaming alternative:
    # total_w = 0.0
    # for e in gv.edges():
    #     total_w += float(w[e])
    # print(f"Total edge weight (filtered): {total_w:,.0f}")
else:
    print(f"No 'weight' edge property found. Total edge weight (filtered) = #edges = {gv.num_edges():,}")


In [ ]:
# --- 3) Connected components + largest connected component size ---
comp, hist = label_components(gv, directed=False)  # respects view
num_cc = int(len(hist))
lcc_id = int(np.argmax(hist)) if num_cc else -1
lcc_size = int(hist[lcc_id]) if num_cc else 0

print(f"Connected components (filtered): {num_cc:,}")
print(f"Largest CC: id={lcc_id}  size={lcc_size:,} vertices")

In [ ]:
# Degree in the ORIGINAL graph g
deg = g.get_total_degrees(g.get_vertices()).astype(np.int64)

Ks = np.arange(3, 20)  # 3..40
lcc_sizes = []
avg_edge_w = []

has_w = "weight" in g.ep
if has_w:
    w = g.ep.weight

for K in Ks:
    print(K, end=', ')
    vfilt = g.new_vertex_property("bool")
    vfilt.a = (deg <= K)

    gv = gt.GraphView(g, vfilt=vfilt)  # no copy

    # LCC size
    if gv.num_vertices() == 0:
        lcc_sizes.append(0)
    else:
        comp, hist = label_components(gv, directed=False)
        lcc_sizes.append(int(hist.max()) if len(hist) else 0)

    # Average edge weight (over edges that remain after filtering)
    m = gv.num_edges()
    if m == 0:
        avg_edge_w.append(0.0)
    else:
        if has_w:
            edges_w = gv.get_edges([w])  # columns: src, dst, weight (respects view)
            total_w = float(edges_w[:, 2].sum()) if edges_w.size else 0.0
            avg_edge_w.append(total_w / m)
        else:
            avg_edge_w.append(1.0)  # no weight property -> treat as all ones

lcc_sizes = np.array(lcc_sizes, dtype=np.int64)
avg_edge_w = np.array(avg_edge_w, dtype=np.float64)

In [ ]:
# Plot: LCC size vs K
plt.figure(figsize=(12, 7))
plt.plot(Ks, lcc_sizes, marker='o')
plt.xlabel("K (keep vertices with degree ≤ K)")
plt.ylabel("Largest connected component size (vertices)")
plt.title("LCC size vs degree threshold K")
plt.grid(True)
plt.show()

# Print table
print("K\tLCC_size\tavg_edge_weight")
for K, s, a in zip(Ks, lcc_sizes, avg_edge_w):
    print(f"{int(K)}\t{s}\t{a:.3f}")

In [ ]:
# Degree in the ORIGINAL graph g
deg = g.get_total_degrees(g.get_vertices()).astype(np.int64)

Ks = np.arange(1, 8)  # K = 1..7
lcc_sizes = []

for K in Ks:
    print(K, end=', ')
    # Keep vertices with K <= degree <= 30
    vfilt = g.new_vertex_property("bool")
    vfilt.a = (deg >= K) & (deg <= 30)

    gv = gt.GraphView(g, vfilt=vfilt)  # no copy

    if gv.num_vertices() == 0 or gv.num_edges() == 0:
        lcc_sizes.append(0)
        continue

    _, hist = label_components(gv, directed=False)
    lcc_sizes.append(int(hist.max()) if len(hist) else 0)
    
    print(hist.max())

lcc_sizes = np.array(lcc_sizes, dtype=np.int64)

In [ ]:
# Plot
plt.figure(figsize=(12, 7))
plt.plot(Ks, lcc_sizes, marker='o')
plt.xlabel("Lower degree bound K (keep K ≤ degree ≤ 30)")
plt.ylabel("Largest connected component size (vertices)")
plt.title("LCC size for vertex-degree band [K, 30]")
plt.xticks(Ks)
plt.grid(True)
plt.show()

# Optional print
for K, s in zip(Ks, lcc_sizes):
    print(f"K={int(K)}  LCC size={s:,}")

In [ ]:
# Requires an edge property named "weight"
if "weight" not in g.ep:
    raise RuntimeError("Graph has no edge property 'weight'")

w = g.ep.weight

Ts = np.arange(1, 10)  # thresholds 1..10
lcc_sizes = []

for T in Ts:
    print(T, end=', ')
    # Keep only edges with weight >= T (NO COPY)
    efilt = g.new_edge_property("bool")
    efilt.a = (w.a >= T)

    gv = gt.GraphView(g, efilt=efilt)  # filtered view

    if gv.num_vertices() == 0 or gv.num_edges() == 0:
        lcc_sizes.append(0)
        continue

    comp, hist = label_components(gv, directed=False)
    lcc_sizes.append(int(hist.max()) if len(hist) else 0)

lcc_sizes = np.array(lcc_sizes, dtype=np.int64)


In [ ]:
# Plot
plt.figure(figsize=(12, 7))
plt.plot(Ts, lcc_sizes, marker='o')
plt.xlabel("Edge weight threshold T (keep edges with weight ≥ T)")
plt.ylabel("Largest connected component size (vertices)")
plt.title("LCC size vs edge weight threshold")
plt.grid(True)
plt.show()

# Optional: print values
for T, s in zip(Ts, lcc_sizes):
    print(f"T={int(T):2d}  LCC size={s:,}")

In [ ]:
import numpy as np

def print_edge_weight_stats(g):
    """
    Prints counts + percentages of edges by weight bins:
    1,2,3,4,5,6,7,8,9,10, and >10.
    """
    if "weight" not in g.ep:
        raise RuntimeError("Graph has no edge property 'weight'")

    w_all = g.ep.weight
    edges_w = g.get_edges([w_all])  # respects GraphView filters
    if edges_w.size == 0:
        print("Total edges: 0")
        return

    w = edges_w[:, 2]
    M = int(w.size)

    def _count(mask):
        return int(np.count_nonzero(mask))

    def _pct(c):
        return 100.0 * c / M if M else 0.0

    bins = [
        ("w = 1",   (w == 1)),
        ("w = 2",   (w == 2)),
        ("w = 3",   (w == 3)),
        ("w = 4",   (w == 4)),
        ("w = 5",   (w == 5)),
        ("w = 6",   (w == 6)),
        ("w = 7",   (w == 7)),
        ("w = 8",   (w == 8)),
        ("w = 9",   (w == 9)),
        ("w = 10",  (w == 10)),
        ("w = 11",  (w == 11)),
        ("w = 12",  (w == 12)),
        ("w = 13",  (w == 13)),
        ("w = 14",  (w == 14)),
        ("w = 15",  (w == 15)),
        ("w = 16",  (w == 16)),
        ("w = 17",  (w == 17)),
        ("w = 18",  (w == 18)),
        ("w = 19",  (w == 19)),
        ("w = 20",  (w == 20)),
        ("w > 20",  (w > 20)),
        ("w > 30",  (w > 30)),
        ("w > 40",  (w > 40)),
        ("w > 50",  (w > 50)),
        ("w > 100",  (w > 100)),
        ("w > 150",  (w > 150)),
        ("w > 200",  (w > 200)),
    ]

    print(f"Total edges: {M:,}\n")
    for label, mask in bins:
        c = _count(mask)
        print(f"{label:6s}: {c:>10,}  ({_pct(c):6.2f}%)")


In [ ]:
print_edge_weight_stats(g)

## Filter and save
Keep edges with weight >= `min_w`, take the LCC, drop vertices with degree > `max_deg`, iteratively strip degree-1 leaves, take the LCC again, and save. The saved file is the input to step 06a or step 07.

In [ ]:
def filter_lcc_degcap_leafstrip_prune_save(
    g,
    weight_ep="weight",
    min_w=2,
    max_deg=30,
    out_path="g_wge2_LCC_degLE30_leafstripped_pruned.gt.gz",
    directed=False,
):
    w = g.ep[weight_ep] if isinstance(weight_ep, str) else weight_ep

    # Edge filter: weight >= min_w
    ef = g.new_ep("bool")
    ef.a = (w.a >= min_w)
    gv = gt.GraphView(g, efilt=ef, directed=directed)

    # LCC
    comp, hist = label_components(gv)
    if len(hist) == 0 or int(hist.max()) == 0:
        raise RuntimeError("Empty after weight filter.")
    lab = int(hist.argmax())
    vf = gv.new_vp("bool"); vf.a = (comp.a == lab)
    cur = gt.GraphView(gv, vfilt=vf)
    print(f"After w>={min_w} + LCC: {cur.num_vertices():,} V, {cur.num_edges():,} E")

    # Vertex filter: degree <= max_deg (degree computed in current view)
    deg = cur.degree_property_map("total")
    vf = cur.new_vp("bool"); vf.a = (deg.a <= max_deg)
    cur = gt.GraphView(cur, vfilt=vf)
    print(f"After degree<={max_deg}: {cur.num_vertices():,} V, {cur.num_edges():,} E")

    # Iteratively remove degree == 1 vertices until none left
    while True:
        deg = cur.degree_property_map("total")
        n_deg1 = int((deg.a == 1).sum())
        if n_deg1 == 0:
            break
        vf = cur.new_vp("bool")
        vf.a = (deg.a != 1)          # exact: remove degree==1, keep others (including degree 0)
        cur = gt.GraphView(cur, vfilt=vf)
        print(f"  leaf-strip: removed {n_deg1:,} deg==1 -> {cur.num_vertices():,} V, {cur.num_edges():,} E")

    # LCC again (leaf stripping / degree cap can disconnect)
    comp, hist = label_components(cur)
    if len(hist) == 0 or int(hist.max()) == 0:
        raise RuntimeError("Empty after leaf stripping.")
    lab = int(hist.argmax())
    vf = cur.new_vp("bool"); vf.a = (comp.a == lab)
    cur = gt.GraphView(cur, vfilt=vf)
    print(f"Final LCC: {cur.num_vertices():,} V, {cur.num_edges():,} E")
    
    # NEW: total edge weight in the final LCC (view)
    total_w = sum(float(w[e]) for e in cur.edges())
    print(f"Final LCC total edge weight: {total_w:,.3f}")

    out_g = gt.Graph(cur, prune=True)
    out_g.save(out_path)
    print(f"Saved to {out_path}")
    return out_g

In [ ]:
min_w = 3
max_deg = 30

out_filename = f"{scratch_prefix}{SAMPLE_ID}_barcodesGraph_postPipeline0302_Lev3_min_e{min_w}_min_d2_max_d{max_deg}_LCC.gt.gz"
lcc = filter_lcc_degcap_leafstrip_prune_save(g, min_w=min_w, max_deg=max_deg, out_path=out_filename)